# Build a model to predict Salary on the base of age , experience , department  and position

# Import Library

In [81]:
import pandas as pd 
import numpy as np

# Load DataSet

In [82]:
data = pd.read_csv("knn_regression_complex_dataset.csv")

In [83]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Age         600 non-null    int64 
 1   Experience  600 non-null    int64 
 2   Department  600 non-null    object
 3   Position    600 non-null    object
 4   Education   600 non-null    object
 5   Salary      600 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 28.3+ KB


In [84]:
data.head()

,Age,Experience,Department,Position,Education,Salary
0,59,28,IT,Director,Bachelors,212620
1,49,12,Operations,Manager,Bachelors,111267
2,35,34,Operations,Manager,Masters,206352
3,28,5,Finance,Director,Masters,107703
4,41,17,IT,VP,Masters,189429


In [85]:
data['Department'].unique()

array(['IT', 'Operations', 'Finance', 'Sales', 'HR'], dtype=object)

In [86]:
data['Position'].unique()

array(['Director', 'Manager', 'VP', 'Consultant', 'Analyst'], dtype=object)

# Features and label sepration

In [87]:
features = data.iloc[:,[0,1,2,3]].values
label = data.iloc[:,5].values

# One Hot Encoding

In [88]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Age         600 non-null    int64 
 1   Experience  600 non-null    int64 
 2   Department  600 non-null    object
 3   Position    600 non-null    object
 4   Education   600 non-null    object
 5   Salary      600 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 28.3+ KB


In [89]:
from sklearn.preprocessing import OneHotEncoder

oheState = OneHotEncoder(sparse_output=False)
department = oheState.fit_transform(features[:,2].reshape(-1,1))
position = oheState.fit_transform(features[:,3].reshape(-1,1))

In [90]:
department

array([[0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.],
       ...,
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0.]], shape=(600, 5))

In [91]:
position

array([[0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.],
       ...,
       [0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.]], shape=(600, 5))

# Mearge department encoding and position encoding

In [92]:
final_features = np.concatenate((department,features[:,[0,1]],position),axis=1)

# Feature Selection 

In [93]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

In [94]:
algo = LinearRegression()
sfm = SelectFromModel(estimator=algo)

In [95]:
sfm.fit(final_features,label)

,estimator,LinearRegression()
,threshold,None
,prefit,False
,norm_order,1
,max_features,None
,importance_getter,'auto'
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [96]:
sfm.get_support()

array([False, False, False, False, False, False, False,  True,  True,
        True, False,  True])

# RFE

In [97]:
from sklearn.feature_selection import RFE

In [98]:
rfe = RFE(estimator=algo)
rfe.fit(final_features,label)

,estimator,LinearRegression()
,n_features_to_select,None
,step,1
,verbose,0
,importance_getter,'auto'
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [99]:
rfe.ranking_

array([3, 6, 7, 5, 2, 4, 1, 1, 1, 1, 1, 1])

# Model Building

In [100]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

In [101]:
for rs in range(1,301):
    x_train,x_test,y_train,y_test = train_test_split(final_features,label,test_size=0.2,random_state=rs)

    for k in range(3,11):
        model = KNeighborsRegressor(n_neighbors=k)
        
        model.fit(x_train,y_train)
        
        trainScore = model.score(x_train,y_train)
        testScore = model.score(x_test,y_test)
        
        
        if testScore > trainScore and testScore >= 0.5:
            print(f"Test Score {testScore} and Train Score {trainScore}, k {k} , rs {rs}")

Test Score 0.5763389303951534 and Train Score 0.5645801210512336, k 10 , rs 3
Test Score 0.5702962500979749 and Train Score 0.5682226265682965, k 10 , rs 22
Test Score 0.6419446788609355 and Train Score 0.6209123951723892, k 6 , rs 69
Test Score 0.6536836631673446 and Train Score 0.6105062485888507, k 7 , rs 69
Test Score 0.6325108935878578 and Train Score 0.5862566543037488, k 8 , rs 69
Test Score 0.6154610452517799 and Train Score 0.5681651549938325, k 9 , rs 69
Test Score 0.6146007017059708 and Train Score 0.559883142264168, k 10 , rs 69
Test Score 0.5819166642363626 and Train Score 0.575849470128879, k 10 , rs 72
Test Score 0.5868613936768108 and Train Score 0.5752337320739025, k 10 , rs 141
Test Score 0.5784319568958624 and Train Score 0.5762039770169234, k 10 , rs 166
Test Score 0.5878206820883898 and Train Score 0.5742298470358952, k 9 , rs 172
Test Score 0.5725525089477517 and Train Score 0.5607557854994192, k 10 , rs 172
Test Score 0.5768562247663732 and Train Score 0.57620324

In [102]:
# Test Score 0.6536836631673446 and Train Score 0.6105062485888507, k 7 , rs 69

x_train,x_test,y_train,y_test = train_test_split(final_features,label,test_size=0.2,random_state=69)

model = KNeighborsRegressor(n_neighbors=7)

model.fit(x_train,y_train)
trainScore = model.score(x_train,y_train)
testScore = model.score(x_test,y_test)

In [103]:
from sklearn.metrics import r2_score

In [104]:
y_pred = model.predict(x_test)

r2 = r2_score(y_test,y_pred)
print(r2)
if r2 >= 0.5:
    print("Approve Model")
else:
    print("Improve Model")

0.6536836631673446
Approve Model
